# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# Fetch the list of record sets in the metadata
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets defined in 'metadata'. Trying to auto-detect by loading records...")
    # Sometimes recordSets are not listed in metadata, but records can still be accessed if only one exists
    # We'll try to load from the default record set
    try:
        sample_gen = dataset.records()
        sample = next(sample_gen)
        print("Sample record:", sample)
        # Extract columns from sample dict
        default_record_set_id = dataset._default_record_set
        print("Default record set @id:", default_record_set_id)
        # Fetch possible field/column IDs
        df_columns = list(sample.keys())
        print("Columns (@id):")
        for col in df_columns:
            print(f"  - {col}")
    except Exception as e:
        print("Could not auto-load any records. Error:", e)
else:
    print("Record sets present in dataset:")
    for rs in record_sets:
        print(f"- @id: {rs['@id']}")
        if 'field' in rs:
            fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
            print("  fields (@id):")
            for f in fields:
                if isinstance(f, dict) and '@id' in f:
                    print(f"    - {f['@id']}")
                else:
                    print(f"    - {f}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Detect default record set if metadata did not explicitly define any
try:
    # mlcroissant>=0.5.0 exposes _default_record_set after loading the first record
    # Try to get the @id using sample record as above
    record_set_id = getattr(dataset, '_default_record_set', None)
    if not record_set_id:
        # Trigger first record load (also initializes _default_record_set)
        sample = next(dataset.records())
        record_set_id = dataset._default_record_set
    print(f"Using record set @id: {record_set_id}")
except Exception as e:
    raise RuntimeError(f"Could not auto-detect record set @id: {e}")

# For this dataset, only one record set is likely present:
record_sets = [record_set_id]
dataframes = {}

for rsid in record_sets:
    df = pd.DataFrame(list(dataset.records(record_set=rsid)))
    dataframes[rsid] = df

print(f"Columns in DataFrame loaded from record set {record_set_id}:")
print(dataframes[record_set_id].columns.tolist())
dataframes[record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates outlier filtering, field normalization, and grouping.

In [ ]:
# Display number of rows and data types
df = dataframes[record_set_id]
print(f"Number of rows: {len(df)}")
print(f"Data types:\n{df.dtypes}")

# Try to detect a numeric field (example: 'age', 'interval', or similar columns)
import numpy as np

# Heuristically select a numeric field
numeric_field_candidates = [c for c in df.columns if (df[c].dtype in [int, float, np.int64, np.float64] or pd.api.types.is_numeric_dtype(df[c]))]
if not numeric_field_candidates:
    # Try parsing any columns containing age/interval/numeric in name
    for c in df.columns:
        if any(kw in c.lower() for kw in ['age', 'interval', 'years', 'count', 'number']):
            try:
                df[c] = pd.to_numeric(df[c], errors='coerce')
            except Exception:
                pass
    numeric_field_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]

if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]
    print(f"Selected numeric field for analysis: {numeric_field}")
else:
    raise RuntimeError("Could not find a numeric field in this dataset for EDA.")

# Set analysis threshold (use quantile for meaningful filtering)
try:
    v10 = df[numeric_field].quantile(0.1)
    threshold = v10 if not np.isnan(v10) else df[numeric_field].median()
except Exception:
    threshold = 10

filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

norm_col = f"{numeric_field}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, norm_col]].head())

# Detect a possible grouping/categorical field (e.g., 'sex', 'site', etc.)
group_field_candidates = [c for c in df.columns if (df[c].dtype == object or pd.api.types.is_string_dtype(df[c])) and df[c].nunique() < df.shape[0] // 2 and df[c].nunique() > 1]
group_field = group_field_candidates[0] if group_field_candidates else None

if group_field:
    print(f"Grouping by {group_field}:")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(grouped_df)
else:
    print("Could not automatically select a grouping field. Consider grouping by a relevant categorical variable.")

## 5. Visualization
Visualize distributions or relationships between selected fields using matplotlib and seaborn (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram for the numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field].dropna(), bins=15, kde=True, color='skyblue')
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If grouping field is present, boxplot numeric field by group
if group_field:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()
else:
    print("No grouping field available for boxplot.")

## 6. Conclusion
In this notebook, we have loaded and explored the FAIR^2 dataset using `mlcroissant`. We examined the available record sets, loaded tabular data by `@id`, performed exploratory data analysis and basic visualizations based on detected numeric and grouping fields.

This workflow can be extended for deeper clinical or biomarker analyses, and advanced preprocessing or modeling, fully leveraging the structured Croissant schema for reproducibility and clarity.